# 🎙️ Fish Speech — Podcast Studio

**Multi-speaker long-form podcast generation powered by Fish Audio S2-Pro**

This notebook sets up and launches the Podcast Studio — a Gradio WebUI for generating
podcast-style audio with up to 4 custom cloned voices.

---

### What you can do
- **Clone up to 4 voices** by uploading 5–30 second reference audio clips
- **Write a podcast script** in plain `SpeakerName: text` dialogue format
- **Add emotion tags** inline: `[laugh]`, `[excited]`, `[whisper]`, `[pause]`, etc.
- **Preview each speaker's voice** before generating the full podcast
- **Generate unlimited-length podcasts** — the script is automatically batched

### Requirements
- ✅ **GPU runtime** — go to *Settings → Accelerator → GPU T4 x2* (or T4)
- ✅ ~10 GB disk space for model weights (downloaded automatically)
- ✅ No local setup required

---
Run each cell in order (▶ or Shift+Enter).

## ① Environment Setup

In [ ]:
# ── Check GPU ──────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:', result.stdout.strip())
else:
    print('⚠️  No NVIDIA GPU found. Generation will be very slow on CPU.')
    print('   Go to Settings → Accelerator → GPU T4 x2 to enable a GPU.')

In [ ]:
# ── 1. Detect CUDA version → choose the right PyTorch wheel index ──────────
# Kaggle T4 is usually CUDA 12.4; A100/L4 may be 12.2 or higher.
import subprocess, sys

def _cuda_tag():
    r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    out = r.stdout + r.stderr
    for ver, tag in [('12.9','cu129'),('12.8','cu128'),('12.6','cu126'),
                     ('12.4','cu124'),('12.2','cu122'),('12.1','cu121')]:
        if f'release {ver}' in out:
            return tag
    return 'cu124'  # safe default for modern Kaggle

cuda_tag = _cuda_tag()
torch_index = f'https://download.pytorch.org/whl/{cuda_tag}'
print(f'CUDA tag  : {cuda_tag}')
print(f'Torch idx : {torch_index}')

# ── 2. PyTorch: reuse Kaggle pre-installed build if it's already 2.x ──────
_skip_torch = False
try:
    import torch as _t
    if int(_t.__version__.split('.')[0]) >= 2:
        print(f'Reusing pre-installed torch {_t.__version__} — skipping torch install.')
        _skip_torch = True
except ImportError:
    pass

if not _skip_torch:
    # Try pinned version; silently fall back to latest stable if unavailable.
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch==2.8.0', 'torchaudio==2.8.0', '--index-url', torch_index],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(f'torch==2.8.0 not available on {cuda_tag} — installing latest stable...')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'torch', 'torchaudio', '--index-url', torch_index],
            check=True,
        )
    else:
        print('torch==2.8.0 installed.')

# ── 3. Remaining fish-speech dependencies ──────────────────────────────────
# pyaudio is intentionally omitted — it requires the system PortAudio library
# (apt portaudio19-dev) and is NOT used by the Podcast Studio.
# If you ever need it: !apt-get install -y portaudio19-dev && pip install pyaudio
other_deps = [
    'transformers>=4.40.0', 'datasets>=2.18.0', 'lightning>=2.1.0',
    'hydra-core>=1.3.2', 'einops>=0.7.0', 'einx>=0.2.2',
    'librosa>=0.10.1', 'rich>=13.5.3', 'gradio>5.0.0',
    'loguru>=0.6.0', 'loralib>=0.1.2', 'pyrootutils>=1.0.4',
    'resampy>=0.4.3', 'zstandard>=0.22.0', 'pydub',
    'modelscope>=1.17.1', 'opencc-python-reimplemented>=0.1.7',
    'silero-vad', 'ormsgpack', 'tiktoken>=0.8.0', 'pydantic>=2.11.0',
    'cachetools', 'safetensors', 'openai-whisper', 'soundfile', 'ffmpeg-python',
    'natsort>=8.4.0', 'protobuf>=4.25.1', 'argbind', 'pyloudnorm',
]

print('Installing core dependencies…')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *other_deps])

print('Installing audio codec tools (bypassing protobuf conflicts)…')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'descript-audio-codec', 'descript-audiotools'
])

print('✅ All dependencies installed.')

## ② Clone Repository

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/kaggle/working/fish-speech')

if REPO_DIR.exists():
    print(f'Repository already exists at {REPO_DIR} — pulling latest changes.')
    !git -C {REPO_DIR} pull --ff-only
else:
    print('Cloning fish-speech repository…')
    !git clone https://github.com/fishaudio/fish-speech.git {REPO_DIR}

# Add repo root to Python path so imports work
import sys
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

os.chdir(REPO_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')

## ③ Download Model Weights

The Fish Audio S2-Pro weights (~8 GB) are downloaded from Hugging Face.
This only needs to run once — the files are cached at `checkpoints/s2-pro`.

In [ ]:
from pathlib import Path

CHECKPOINT_DIR = Path('checkpoints/s2-pro')

# Check if weights already exist
weight_files = list(CHECKPOINT_DIR.glob('*.safetensors')) + list(CHECKPOINT_DIR.glob('model.pth'))
codec_file = CHECKPOINT_DIR / 'codec.pth'

if weight_files and codec_file.exists():
    print(f'✅ Model weights already present at {CHECKPOINT_DIR}')
    print(f'   Found: {[f.name for f in weight_files + [codec_file]]}')
else:
    print('Downloading Fish Audio S2-Pro weights from Hugging Face (~8 GB)…')
    print('This may take 5-15 minutes depending on your connection.')

    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='fishaudio/s2-pro',
        local_dir=str(CHECKPOINT_DIR),
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*'],
    )
    print(f'\n✅ Model downloaded to {CHECKPOINT_DIR}')

# Verify essential files
assert codec_file.exists(), f'codec.pth not found in {CHECKPOINT_DIR}'
print('\n✅ Model checkpoint verified.')

## ④ (Optional) Download Whisper for Auto-Transcription

Skip this cell if you plan to type the reference text manually.
Whisper enables the **🎤 Transcribe** button in the Speakers tab.

In [ ]:
# @param {type: "boolean"}
DOWNLOAD_WHISPER = True  # Set to False to skip

WHISPER_DIR = Path('checkpoints/whisper-small')

if DOWNLOAD_WHISPER:
    if (WHISPER_DIR / 'small.pt').exists():
        print(f'✅ Whisper model already at {WHISPER_DIR}')
    else:
        print('Downloading Whisper small model (~244 MB)…')
        import whisper
        WHISPER_DIR.mkdir(parents=True, exist_ok=True)
        # This downloads and caches to WHISPER_DIR
        whisper.load_model('small', download_root=str(WHISPER_DIR))
        print(f'✅ Whisper downloaded to {WHISPER_DIR}')

    WHISPER_MODEL_DIR = str(WHISPER_DIR)
else:
    WHISPER_MODEL_DIR = None
    print('Whisper skipped — transcription buttons will be disabled.')

print(f'WHISPER_MODEL_DIR = {WHISPER_MODEL_DIR}')

## ⑤ Load Models

This loads the S2-Pro LLaMA language model and the DAC audio codec into GPU memory.
Expect ~8–10 GB VRAM usage. The first cell takes ~60 seconds.

In [ ]:
import os, sys, time
os.environ['EINX_FILTER_TRACEBACK'] = 'false'

import torch
from pathlib import Path
from loguru import logger

CHECKPOINT_DIR = Path('checkpoints/s2-pro')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PRECISION = torch.bfloat16  # bfloat16 is best for A100/T4; use torch.half for V100

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── LLaMA (text-to-semantic) model ────────────────────────────────────────
from fish_speech.models.text2semantic.inference import launch_thread_safe_queue

print('\nLoading LLaMA model (this takes ~30-60 s)…')
t0 = time.time()
llama_queue = launch_thread_safe_queue(
    checkpoint_path=CHECKPOINT_DIR,
    device=DEVICE,
    precision=PRECISION,
    compile=False,  # Set True for faster steady-state throughput (first run slow)
)
print(f'✅ LLaMA loaded in {time.time()-t0:.1f}s')

# ── DAC audio codec ───────────────────────────────────────────────────────
from fish_speech.models.dac.inference import load_model as load_decoder_model

print('\nLoading DAC codec…')
t0 = time.time()
decoder_model = load_decoder_model(
    config_name='modded_dac_vq',
    checkpoint_path=CHECKPOINT_DIR / 'codec.pth',
    device=DEVICE,
)
print(f'✅ DAC codec loaded in {time.time()-t0:.1f}s')

# ── Inference engine ─────────────────────────────────────────────────────
from fish_speech.inference_engine import TTSInferenceEngine

inference_engine = TTSInferenceEngine(
    llama_queue=llama_queue,
    decoder_model=decoder_model,
    precision=PRECISION,
    compile=False,
)
print('✅ TTSInferenceEngine created.')

In [ ]:
# ── Warm-up pass: eliminates first-request latency ────────────────────────
from fish_speech.utils.schema import ServeTTSRequest
import time

print('Running warm-up pass…')
t0 = time.time()
list(
    inference_engine.inference(
        ServeTTSRequest(
            text='<|speaker:0|>Hello, warm-up complete.',
            references=[],
            max_new_tokens=128,
            chunk_length=200,
            top_p=0.8,
            repetition_penalty=1.1,
            temperature=0.8,
        )
    )
)
print(f'✅ Warm-up done in {time.time()-t0:.1f}s — model is ready.')

## ⑥ Launch Podcast Studio

Run the cell below to start the Gradio WebUI.
A **public share link** is created automatically so you can open it in any browser.

### How to use the UI

| Tab | What to do |
|-----|------------|
| **🎤 Speakers** | Set how many speakers (1–4). For each: enter a name, upload a 5–30s reference audio clip, provide its transcription (or click 🎤 Transcribe), then click **▶ Preview Voice** to verify the clone. |
| **📝 Script** | Write your podcast in `Name: text` format. Click **🔍 Preview Script Parsing** to confirm speaker names match. |
| **⚙️ Settings** | Adjust temperature, repetition penalty, chunk size, etc. Defaults work well. |
| **Generate** | Click **🎙️ Generate Podcast** — the full audio appears below when done. |

> **Tip:** The model processes the script in batches of `chunk_length` bytes (~200 bytes
> ≈ 3–4 short turns).  Long podcasts (100+ turns) will take several minutes.
> A T4 GPU generates roughly 3–5 seconds of audio per second of real time.

In [ ]:
from podcast.engine import PodcastSynthesizer
from podcast.app import build_podcast_app

# Create the synthesizer wrapper
synthesizer = PodcastSynthesizer(
    tts_engine=inference_engine,
    whisper_model_dir=WHISPER_MODEL_DIR,  # None if Whisper was skipped
)

# Build the Gradio app
app = build_podcast_app(
    synthesizer=synthesizer,
    whisper_model_dir=WHISPER_MODEL_DIR,
)

# Launch with a public share link (required for Kaggle)
app.launch(share=True, quiet=False)

## 📚 Tips & Tricks

### Voice Cloning Quality
- Use **10–20 seconds** of clean, single-speaker audio for best results.
- Avoid background music or noise in the reference clip.
- The reference text must be the **exact transcription** — including filler words.
- Use the **▶ Preview Voice** button to verify cloning before generating the full podcast.

### Script Writing
- Keep individual turns **under ~80 words** for best prosody.
- Add emotion tags to specific words/phrases: `This is [whisper]really[/end] important.`
- Use `[pause]` or `[short pause]` for natural breathing room.
- The script parser is **case-insensitive** for speaker names.

### Generation Speed (T4 GPU)
| Podcast length | Approx. generation time |
|---------------|------------------------|
| 1 min (~150 words) | ~30–60 s |
| 5 min (~750 words) | ~3–5 min |
| 15 min (~2250 words) | ~10–15 min |

### Saving Output
The generated audio can be downloaded directly from the Gradio player (download icon).
To save programmatically, add a cell after launch:

```python
import soundfile as sf
import numpy as np

# After generating, call synthesizer.synthesize_podcast() directly:
sr, audio = synthesizer.synthesize_podcast(
    script=open('my_script.txt').read(),
    speakers=speakers,   # list of SpeakerConfig objects
)
sf.write('podcast_output.wav', audio, sr)
```

### Multiple Sessions
The Kaggle session expires after ~12 hours (free) or 30 hours (with background runs).  
Model weights are cached in `checkpoints/` — re-running cells ③–⑥ reloads them quickly.

## 🐍 Programmatic API (Advanced)

You can also generate podcasts directly from Python without the UI.

In [ ]:
# ── Example: generate a 2-speaker podcast programmatically ────────────────
# (Run this cell AFTER running cells ① – ⑤)

import soundfile as sf
from podcast.engine import SpeakerConfig, PodcastSynthesizer

# --- Define speakers ---
# Use reference audio bytes for voice cloning.
# Here we use no reference (generic voices) for the demo.
speakers = [
    SpeakerConfig(
        name='Alice',
        speaker_id=0,
        reference_audio=None,   # Replace with open('alice_ref.wav','rb').read()
        reference_text='',      # Replace with the exact transcription
    ),
    SpeakerConfig(
        name='Bob',
        speaker_id=1,
        reference_audio=None,
        reference_text='',
    ),
]

# --- Write the script ---
script = """
Alice: Welcome to the show! Today we're talking about the future of AI.
Bob: [excited] I've been looking forward to this conversation all week.
Alice: Let's start with the big question — will AI replace programmers?
Bob: [pause] That's the million-dollar question, isn't it?
Alice: [laugh] More like a trillion-dollar question at this point!
Bob: Fair point. I think the honest answer is: it depends on how we define 'replace'.
Alice: That sounds like a diplomatic answer, Bob.
Bob: [chuckle] Guilty as charged. But seriously — AI will change programming dramatically.
Alice: Well, thanks for joining us today. Until next time!
Bob: Thanks everyone — keep exploring!
""".strip()

# --- Generate ---
print('Generating podcast…')
sample_rate, audio_array = synthesizer.synthesize_podcast(
    script=script,
    speakers=speakers,
    temperature=0.8,
    top_p=0.8,
    repetition_penalty=1.1,
    chunk_length=200,
)

duration = len(audio_array) / sample_rate
print(f'✅ Generated {duration:.1f}s of audio at {sample_rate} Hz')

# --- Save ---
sf.write('/kaggle/working/podcast_output.wav', audio_array, sample_rate)
print('Saved to /kaggle/working/podcast_output.wav')

# --- Play inline ---
from IPython.display import Audio, display
display(Audio(audio_array, rate=sample_rate, autoplay=False))